Preproses and load old model

In [1]:
from torchvision.io import decode_image
from torchvision import datasets, transforms
from torchvision.models import mnasnet0_5, MNASNet0_5_Weights
from torch.utils.data import TensorDataset, DataLoader
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn

device = torch.device("cuda")

data_dir = "C:\\Users\\sirbu\\Downloads\\DS340\\personal\\DOES"
#img = decode_image("test/assets/encode_jpeg/grace_hopper_517x606.jpg")

"""def load_split_train_test(datadir, valid_size = .2):
    train_transforms = transforms.Compose([transforms.Resize(224),
                                       transforms.ToTensor(),
                                       ])
    test_transforms = transforms.Compose([transforms.Resize(224),
                                      transforms.ToTensor(),
                                      ])
    train_data = datasets.ImageFolder(datadir,       
                    transform=train_transforms)
    test_data = datasets.ImageFolder(datadir,
                    transform=test_transforms)
    num_train = len(train_data)
    indices = list(range(num_train))
    split = int(np.floor(valid_size * num_train))
    np.random.shuffle(indices)
    from torch.utils.data.sampler import SubsetRandomSampler
    train_idx, test_idx = indices[split:], indices[:split]
    train_sampler = SubsetRandomSampler(train_idx)
    test_sampler = SubsetRandomSampler(test_idx)
    trainloader = torch.utils.data.DataLoader(train_data,
                   sampler=train_sampler, batch_size=64)
    testloader = torch.utils.data.DataLoader(test_data,
                   sampler=test_sampler, batch_size=64)
    return trainloader, testloader
trainloader, testloader = load_split_train_test(data_dir, .2)
print(trainloader.dataset.classes)"""

# Step 1: Initialize model with the best available weights
weights = MNASNet0_5_Weights.DEFAULT
model = mnasnet0_5(weights=weights)

model.classifier = nn.Sequential(
    nn.Flatten(),
    nn.Dropout(0.5),
    nn.Linear(1280, 128),
    nn.ReLU(),
    nn.Dropout(0.25),
    nn.Linear(128, 8),  # 10 classes in STL10
    nn.Softmax(dim=1)
)


# Step 2: Initialize the inference transforms
preprocess = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(.5),
    transforms.RandomRotation(degrees=[1,30]),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], # Some common mean RGB values
                         std=[0.229, 0.224, 0.225])  # Common stddevs for RGB values

])
train_dataset = datasets.ImageFolder("C:\\Users\\sirbu\\Downloads\\DS340\\personal\\DOES", transform=preprocess)
test_dataset = datasets.ImageFolder("C:\\Users\\sirbu\\Downloads\\DS340\\personal\\TEST", transform=preprocess)
# Step 3: Apply inference preprocessing transforms
train_loader = DataLoader(train_dataset, batch_size=500, shuffle=True,pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=500, shuffle=True,pin_memory=True)
#batch = preprocess(train_dataset).unsqueeze(0)






Test Function

In [2]:
def get_accuracy_and_loss(model, loader, criterion):
  model.eval()
  my_loss = 0
  with torch.no_grad():
    correct = 0
    for data, target in loader:
      data, target = data.to(device), target.to(device)
      output = model(data)
      pred = output.argmax(dim=1)
      correct += pred.eq(target).sum().item()
      my_loss += criterion(output, target).item()
  return correct/len(loader.dataset), my_loss/len(loader.dataset)

Run test

In [3]:
import torch.nn.functional as F
import torch.optim as optim
best_val_loss = 1000000
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)
model.to(device)

train_losses = []
val_losses = []
train_accuracies = []
val_accuracies = []

for epoch in range(1000):
    model.train()
    train_loss = 0
    correct = 0
    total_count = 0
    for data, target in train_loader:
        data, target = data.to(device), target.to(device)
        optimizer.zero_grad()
        output = model(data)
        loss = criterion(output, target)

        train_loss += loss.item()
        pred = output.argmax(dim=1)
        correct += pred.eq(target).sum().item()
        total_count += data.size(0)
        loss.backward()
        optimizer.step()

    print(f"Epoch {epoch} done.")
    #Train Accuracy
    train_accuracy = correct / total_count
    train_accuracies.append(train_accuracy)
    print(f"Train accuracy: {train_accuracy}")

    #Train Loss
    train_loss = train_loss / total_count
    train_losses.append(train_loss)
    print(f"Train loss: {train_loss}")

    #Validation Accuracy and Loss
    val_accuracy, val_loss = get_accuracy_and_loss(model, test_loader, criterion)

    #Accuracy
    print(f"Val accuracy: {val_accuracy}")
    val_accuracies.append(val_accuracy)

    #Loss
    print(f"Val loss: {val_loss}")
    val_losses.append(val_loss)

    if val_loss <= best_val_loss:
        best_val_loss = val_loss
        torch.save({'model_state_dict': model.state_dict(),
            "val_loss": val_loss,
            "train_losses":train_losses,
            "val_losses":val_losses},"best_model.pth")

Epoch 0 done.
Train accuracy: 0.7814530257683042
Train loss: 0.003020580475918089
Val accuracy: 0.48007704345732516
Val loss: 0.0036565762529365132
Epoch 1 done.
Train accuracy: 0.9401872399445215
Train loss: 0.0026800661661283962
Val accuracy: 0.5141446972432888
Val loss: 0.003587819543486972
Epoch 2 done.
Train accuracy: 0.9546408496970582
Train loss: 0.002649783262528419
Val accuracy: 0.5003009510051764
Val loss: 0.003601497370829088
Epoch 3 done.
Train accuracy: 0.9672695087232644
Train loss: 0.002624805303931854
Val accuracy: 0.5556759359576261
Val loss: 0.003508542538894648
Epoch 4 done.
Train accuracy: 0.9735655887291043
Train loss: 0.002611452479877301
Val accuracy: 0.5990128807030215
Val loss: 0.0034246363335027727
Epoch 5 done.
Train accuracy: 0.9762026425286517
Train loss: 0.0026057844845445775
Val accuracy: 0.6019020103527146
Val loss: 0.0034194389118016866
Epoch 6 done.
Train accuracy: 0.9790404409080955
Train loss: 0.0026000062767471987
Val accuracy: 0.6194775490550138
Va

KeyboardInterrupt: 